# LeetCode 605 — Can Place Flowers bằng Python

**Link đề:** https://leetcode.com/problems/can-place-flowers/
**Độ khó:** Easy · **Chủ đề:** Array, Greedy

```python
class Solution:
    def canPlaceFlowers(self, flowerbed: list[int], n: int) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu đúng luật "không hai bông hoa cạnh nhau" và các **ô biên** (đầu/cuối luồng hoa).
2. Làm quen tư duy **tham lam (greedy)**: gặp chỗ trồng được là trồng ngay – và **chứng minh** vì sao làm vậy không thiệt.
3. Viết 4 lời giải: brute force thử mọi cách (oracle) → tham lam kiểm tra hàng xóm → tham lam với **đệm (padding)** → **đếm đoạn số 0** bằng công thức.
4. Xử lý biên gọn gàng bằng `i == 0 or ...`, kỹ thuật đệm `[0] + bed + [0]`, và dừng sớm.
5. Trace, invariant, Time/Space, lỗi hay gặp, 6 bài tập có gợi ý và đáp án cuối.

## 1. Đề bài và ví dụ

Có một luồng hoa dài; một số ô đã trồng, số khác còn trống. Luật: **không được trồng hai bông ở hai ô kề nhau**.

Cho mảng `flowerbed` gồm `0` (trống) và `1` (đã trồng), và số nguyên `n`. Trả `True` nếu có thể trồng **thêm `n` bông** mà không vi phạm luật, ngược lại `False`.

| flowerbed | n | Output | Vì sao |
|---|---:|:---:|---|
| `[1,0,0,0,1]` | 1 | `True` | trồng ở giữa: `[1,0,1,0,1]` |
| `[1,0,0,0,1]` | 2 | `False` | chỉ có đúng 1 chỗ |

**Constraints:**
- `1 <= len(flowerbed) <= 2·10^4`; mỗi phần tử là `0` hoặc `1`.
- Ban đầu **không có hai bông kề nhau**.
- `0 <= n <= len(flowerbed)`.

Câu hỏi khởi động: ô **đầu tiên** chỉ có hàng xóm bên phải. Vậy `[0,0,1]` trồng được ở ô 0 không? (Có: ô 0 trống, ô 1 trống, bên trái là "ngoài luồng".)

## 2. Nền tảng Python

### 2.1. Điều kiện một ô trồng được
Ô `i` trồng được ⇔ `bed[i] == 0` **và** hàng xóm trái trống (hoặc không có) **và** hàng xóm phải trống (hoặc không có):

```python
left_empty  = (i == 0) or bed[i - 1] == 0
right_empty = (i == len(bed) - 1) or bed[i + 1] == 0
```

`or` **đoản mạch** (short-circuit): nếu vế trái đã `True` thì **không tính** vế phải. Nhờ đó khi `i == 0` ta không bao giờ đọc `bed[-1]` (trong Python `bed[-1]` là phần tử **cuối** – sai ý nghĩa, không báo lỗi!).

In [1]:
bed = [0, 0, 1]
print("bed[-1] là phần tử cuối:", bed[-1], "<- nếu lỡ dùng làm 'hàng xóm trái' của ô 0 thì sai mà không báo lỗi")

i = 0
left_empty = (i == 0) or bed[i - 1] == 0        # vế phải không được tính
right_empty = (i == len(bed) - 1) or bed[i + 1] == 0
print("ô 0: trái trống?", left_empty, "| phải trống?", right_empty)

bed[-1] là phần tử cuối: 1 <- nếu lỡ dùng làm 'hàng xóm trái' của ô 0 thì sai mà không báo lỗi
ô 0: trái trống? True | phải trống? True


### 2.2. Copy list để không sửa input
`bed[:]` hoặc `list(bed)` tạo **bản sao nông**. Nếu sửa trực tiếp `flowerbed`, người gọi hàm sẽ thấy mảng của họ bị đổi – tránh trong code minh họa (LeetCode thì chấp nhận).

In [2]:
original = [1, 0, 0, 0, 1]
copy = original[:]
copy[2] = 1
print("bản sao:", copy, "| gốc vẫn:", original)

bản sao: [1, 0, 1, 0, 1] | gốc vẫn: [1, 0, 0, 0, 1]


### 2.3. `any` và `all`
- `all(iterable)`: `True` nếu **mọi** phần tử đúng (rỗng → `True`).
- `any(iterable)`: `True` nếu **có ít nhất một** phần tử đúng (rỗng → `False`).
Ta sẽ dùng `all` để kiểm tra "không có hai số 1 kề nhau", và `any` trong một bài tập.

In [3]:
bed = [1, 0, 1, 0, 0]
print("Không có hai số 1 kề nhau?", all(not (bed[i] == 1 and bed[i + 1] == 1) for i in range(len(bed) - 1)))
print("Có ô nào trống?", any(v == 0 for v in bed))
print("all([]) =", all([]), "| any([]) =", any([]))

Không có hai số 1 kề nhau? True
Có ô nào trống? True
all([]) = True | any([]) = False


## 3. Cách 0 — Brute force: thử mọi tập ô để trồng (oracle)

Câu hỏi "có trồng được `n` bông không" ⇔ "số bông **tối đa** trồng thêm được ≥ `n`". Brute force: thử mọi tập con các ô trống (bitmask), giữ những tập hợp lệ, lấy kích thước lớn nhất. `O(2^z · len)` với `z` = số ô trống → chỉ dùng cho mảng nhỏ để **kiểm chứng** cách tham lam.

In [4]:
def max_new_flowers_bruteforce(bed):
    empties = [i for i, v in enumerate(bed) if v == 0]
    best = 0
    for mask in range(1 << len(empties)):
        trial = bed[:]
        for k, pos in enumerate(empties):
            if mask >> k & 1:
                trial[pos] = 1
        ok = all(not (trial[i] == 1 and trial[i + 1] == 1) for i in range(len(trial) - 1))
        if ok:
            best = max(best, bin(mask).count("1"))
    return best

def can_place_bruteforce(bed, n):
    return max_new_flowers_bruteforce(bed) >= n

print(max_new_flowers_bruteforce([1, 0, 0, 0, 1]))     # 1
print(can_place_bruteforce([1, 0, 0, 0, 1], 1), can_place_bruteforce([1, 0, 0, 0, 1], 2))

1
True False


`bin(mask)` cho chuỗi nhị phân như `'0b101'`; `.count("1")` đếm số bit 1 = số ô được chọn. `all(...)` trả `True` nếu mọi phần tử đều đúng.

## 4. Insight tham lam: gặp chỗ trồng được thì trồng ngay

Duyệt từ trái sang phải. Nếu ô `i` trồng được, **trồng luôn** (đặt `bed[i] = 1`) và tăng bộ đếm.

Tại sao không thiệt? Giả sử một cách tối ưu **không** trồng ở `i` (ô trồng được đầu tiên). Nhìn bông đầu tiên của cách tối ưu ở bên phải `i`, tại vị trí `j > i`. Ta "dời" bông đó về `i`: ô `i` hợp lệ (trái trống, phải `i+1` – nếu `i+1 = j` thì giờ trống vì đã dời; nếu `i+1 < j` thì vốn trống). Bên phải `j` không bị ảnh hưởng xấu vì `i < j`. Số bông giữ nguyên → luôn tồn tại cách tối ưu trồng ở `i`. Lặp lại lập luận cho phần còn lại.

Trồng ngay cũng **thay đổi** ô `i` thành `1`, nên ô `i+1` sẽ tự động bị chặn ở bước sau – đó là lý do phải **ghi lại** việc trồng.

## 5. Cách 1 — Tham lam kiểm tra hàng xóm: giảng từng đoạn rồi ghép

- **Bước 1:** `bed = flowerbed[:]`, `count = 0`.
- **Bước 2:** `for i in range(len(bed)):`
- **Bước 3:** nếu `bed[i] == 0` và trái trống và phải trống → `bed[i] = 1; count += 1`.
- **Bước 4:** `return count >= n`.

In [5]:
def can_place_greedy(flowerbed, n):
    bed = flowerbed[:]
    count = 0
    for i in range(len(bed)):
        if bed[i] == 0:
            left_empty = (i == 0) or bed[i - 1] == 0
            right_empty = (i == len(bed) - 1) or bed[i + 1] == 0
            if left_empty and right_empty:
                bed[i] = 1                # trồng ngay
                count += 1
    return count >= n


print(can_place_greedy([1, 0, 0, 0, 1], 1))   # True
print(can_place_greedy([1, 0, 0, 0, 1], 2))   # False
print(can_place_greedy([0], 1))               # True
print(can_place_greedy([0, 0, 0, 0, 0], 3))   # True: trồng ở 0, 2, 4

True
False
True
True


### 5.1. TRACE tham lam
Ví dụ `bed = [0,0,1,0,0,0,0,1,0,0]`.

| i | bed[i] | trái trống? | phải trống? | hành động | count |
|---:|---:|:---:|:---:|---|---:|
| 0 | 0 | có (biên) | có | trồng | 1 |
| 1 | 0 | không (vừa trồng ô 0) | – | bỏ | 1 |
| 2 | 1 | – | – | đã có hoa | 1 |
| 3 | 0 | không | – | bỏ | 1 |
| 4 | 0 | có | có | trồng | 2 |
| 5 | 0 | không | – | bỏ | 2 |
| 6 | 0 | có | không (ô 7 = 1) | bỏ | 2 |
| 7 | 1 | – | – | đã có hoa | 2 |
| 8 | 0 | không | – | bỏ | 2 |
| 9 | 0 | có | có (biên) | trồng | 3 |

In [6]:
def trace_greedy(flowerbed):
    bed = flowerbed[:]
    count = 0
    for i in range(len(bed)):
        if bed[i] == 1:
            print(f"i={i}: đã có hoa")
            continue
        left_empty = (i == 0) or bed[i - 1] == 0
        right_empty = (i == len(bed) - 1) or bed[i + 1] == 0
        if left_empty and right_empty:
            bed[i] = 1
            count += 1
            act = "TRỒNG"
        else:
            act = "bỏ"
        print(f"i={i}: trái trống={left_empty!s:<5} phải trống={right_empty!s:<5} -> {act:<5} | count={count} | bed={bed}")
    return count

trace_greedy([0, 0, 1, 0, 0, 0, 0, 1, 0, 0])

i=0: trái trống=True  phải trống=True  -> TRỒNG | count=1 | bed=[1, 0, 1, 0, 0, 0, 0, 1, 0, 0]
i=1: trái trống=False phải trống=False -> bỏ    | count=1 | bed=[1, 0, 1, 0, 0, 0, 0, 1, 0, 0]
i=2: đã có hoa
i=3: trái trống=False phải trống=True  -> bỏ    | count=1 | bed=[1, 0, 1, 0, 0, 0, 0, 1, 0, 0]
i=4: trái trống=True  phải trống=True  -> TRỒNG | count=2 | bed=[1, 0, 1, 0, 1, 0, 0, 1, 0, 0]
i=5: trái trống=False phải trống=True  -> bỏ    | count=2 | bed=[1, 0, 1, 0, 1, 0, 0, 1, 0, 0]
i=6: trái trống=True  phải trống=False -> bỏ    | count=2 | bed=[1, 0, 1, 0, 1, 0, 0, 1, 0, 0]
i=7: đã có hoa
i=8: trái trống=False phải trống=True  -> bỏ    | count=2 | bed=[1, 0, 1, 0, 1, 0, 0, 1, 0, 0]
i=9: trái trống=True  phải trống=True  -> TRỒNG | count=3 | bed=[1, 0, 1, 0, 1, 0, 0, 1, 0, 1]


3

`continue` bỏ qua phần còn lại của vòng hiện tại, nhảy sang `i` kế tiếp.

### 5.2. Dừng sớm
Khi `count >= n` đã đủ, không cần duyệt tiếp: `return True` ngay. Đặc biệt `n = 0` → `True` lập tức.

## 6. Cách 2 — Kỹ thuật đệm (padding): bỏ hẳn kiểm tra biên

Thêm một ô `0` ở **hai đầu**: `padded = [0] + flowerbed + [0]`. Ô đệm không bao giờ được trồng (ta chỉ duyệt `i` từ `1` tới `len − 2`), nhưng chúng giúp ô thật đầu/cuối luôn có "hàng xóm trống" → điều kiện chỉ còn:

```python
padded[i - 1] == 0 and padded[i] == 0 and padded[i + 1] == 0
```

`+` giữa hai list là **nối list** (tạo list mới).

In [7]:
print([0] + [1, 0, 0] + [0])

def can_place_padding(flowerbed, n):
    padded = [0] + flowerbed + [0]
    count = 0
    for i in range(1, len(padded) - 1):
        if padded[i - 1] == 0 and padded[i] == 0 and padded[i + 1] == 0:
            padded[i] = 1
            count += 1
            if count >= n:
                return True
    return count >= n

print(can_place_padding([1, 0, 0, 0, 1], 1), can_place_padding([1, 0, 0, 0, 1], 2), can_place_padding([0], 1))

[0, 1, 0, 0, 0]
True False True


## 7. Cách 3 — Đếm đoạn số 0 liên tiếp (công thức, không sửa mảng)

Nhìn luồng hoa như các **đoạn số 0** ngăn cách bởi số 1. Với một đoạn `z` số 0:

| Loại đoạn | Ví dụ | Số bông tối đa |
|---|---|---|
| Kẹp giữa hai số 1 | `1 0 0 0 1` (`z=3`) | `(z − 1) // 2` |
| Chạm **một** biên | `0 0 1` (`z=2`) | `z // 2` |
| Chạm **hai** biên (cả mảng toàn 0) | `0 0 0` (`z=3`) | `(z + 1) // 2` |

Mẹo gộp: đệm cả hai đầu bằng `0` thì mọi đoạn đều thành "kẹp giữa" với độ dài tăng thêm phần biên… cách dễ nhớ hơn: coi **biên như có thêm một số 0 ảo**. Đoạn chạm một biên có `z` số 0 ⇒ coi như `z + 1` số 0 kẹp giữa ⇒ `((z + 1) − 1) // 2 = z // 2`. Chạm hai biên ⇒ `z + 2` ⇒ `(z + 1) // 2`. Khớp bảng!

`//` là **chia lấy phần nguyên**: `7 // 2 = 3`.

In [8]:
for z in range(1, 7):
    print(f"z={z}: kẹp giữa {(z - 1) // 2}, chạm 1 biên {z // 2}, chạm 2 biên {(z + 1) // 2}")

z=1: kẹp giữa 0, chạm 1 biên 0, chạm 2 biên 1
z=2: kẹp giữa 0, chạm 1 biên 1, chạm 2 biên 1
z=3: kẹp giữa 1, chạm 1 biên 1, chạm 2 biên 2
z=4: kẹp giữa 1, chạm 1 biên 2, chạm 2 biên 2
z=5: kẹp giữa 2, chạm 1 biên 2, chạm 2 biên 3
z=6: kẹp giữa 2, chạm 1 biên 3, chạm 2 biên 3


In [9]:
def can_place_count_zeros(flowerbed, n):
    total = 0
    zeros = 1                       # số 0 ảo ở biên trái
    for v in flowerbed:
        if v == 0:
            zeros += 1
        else:
            total += (zeros - 1) // 2
            zeros = 0
    zeros += 1                      # số 0 ảo ở biên phải
    total += (zeros - 1) // 2
    return total >= n


print(can_place_count_zeros([1, 0, 0, 0, 1], 1), can_place_count_zeros([1, 0, 0, 0, 1], 2))
print(can_place_count_zeros([0, 0, 1, 0, 0, 0, 0, 1, 0, 0], 3))

True False
True


In [10]:
def trace_count_zeros(flowerbed):
    total, zeros = 0, 1
    print("Bắt đầu: zeros = 1 (số 0 ảo bên trái)")
    for i, v in enumerate(flowerbed):
        if v == 0:
            zeros += 1
            print(f"i={i} v=0 -> zeros={zeros}")
        else:
            add = (zeros - 1) // 2
            total += add
            print(f"i={i} v=1 -> đóng đoạn: ({zeros}-1)//2 = {add}, total={total}, reset zeros=0")
            zeros = 0
    zeros += 1
    add = (zeros - 1) // 2
    total += add
    print(f"Kết thúc: +1 số 0 ảo bên phải -> zeros={zeros}, cộng {add}, total={total}")
    return total

trace_count_zeros([0, 0, 1, 0, 0, 0, 0, 1, 0, 0])

Bắt đầu: zeros = 1 (số 0 ảo bên trái)
i=0 v=0 -> zeros=2
i=1 v=0 -> zeros=3
i=2 v=1 -> đóng đoạn: (3-1)//2 = 1, total=1, reset zeros=0
i=3 v=0 -> zeros=1
i=4 v=0 -> zeros=2
i=5 v=0 -> zeros=3
i=6 v=0 -> zeros=4
i=7 v=1 -> đóng đoạn: (4-1)//2 = 1, total=2, reset zeros=0
i=8 v=0 -> zeros=1
i=9 v=0 -> zeros=2
Kết thúc: +1 số 0 ảo bên phải -> zeros=3, cộng 1, total=3


3

## 8. Code nộp LeetCode

Bản chính: tham lam + dừng sớm (sửa trực tiếp `flowerbed` là chấp nhận được trên LeetCode; nếu muốn an toàn thì copy).

In [11]:
class Solution:
    def canPlaceFlowers(self, flowerbed: list[int], n: int) -> bool:
        if n == 0:
            return True
        length = len(flowerbed)
        for i in range(length):
            if (flowerbed[i] == 0
                    and (i == 0 or flowerbed[i - 1] == 0)
                    and (i == length - 1 or flowerbed[i + 1] == 0)):
                flowerbed[i] = 1
                n -= 1
                if n == 0:
                    return True
        return False


sol = Solution()
assert sol.canPlaceFlowers([1, 0, 0, 0, 1], 1) is True
assert sol.canPlaceFlowers([1, 0, 0, 0, 1], 2) is False
print("Solution PASS ví dụ đề")

Solution PASS ví dụ đề


Bản không sửa input (đếm đoạn 0):

In [12]:
class SolutionCountZeros:
    def canPlaceFlowers(self, flowerbed: list[int], n: int) -> bool:
        total, zeros = 0, 1
        for v in flowerbed:
            if v == 0:
                zeros += 1
            else:
                total += (zeros - 1) // 2
                zeros = 0
        total += zeros // 2          # = ((zeros + 1) - 1) // 2
        return total >= n

assert SolutionCountZeros().canPlaceFlowers([1, 0, 0, 0, 1], 1) is True
assert SolutionCountZeros().canPlaceFlowers([0, 0, 0], 2) is True
print("SolutionCountZeros PASS")

SolutionCountZeros PASS


## 9. Vì sao đúng?

**Tham lam.** Invariant: sau khi xử lý các ô `0..i`, số bông đã trồng `count` bằng **số bông tối đa** có thể trồng thêm trong đoạn `0..i` (với ràng buộc từ các bông có sẵn), **và** trong mọi cách đạt tối đa đó, cách của ta là cách "đẩy về trái nhất" – để lại nhiều chỗ nhất cho phần bên phải (bông cuối cùng của ta ở vị trí nhỏ nhất có thể).
- Lập luận "dời bông" ở mục 4 cho thấy trồng ở ô hợp lệ đầu tiên không làm giảm tối đa.
- Bông của ta ở càng trái thì càng ít chặn ô `i+1` → phần bên phải không bao giờ bị thiệt.

**Đếm đoạn 0.** Các đoạn số 0 độc lập với nhau (bị ngăn bởi số 1 có sẵn). Trong một đoạn kẹp giữa dài `z`, hai ô sát hai số 1 không trồng được; còn lại `z − 2` ô liên tiếp tự do, trồng tối đa `ceil((z − 2) / 2) = (z − 1) // 2` (xen kẽ). Biên = thêm số 0 ảo như mục 7.

**Dừng sớm đúng** vì `count` chỉ tăng; đã `>= n` thì kết quả cuối chắc chắn `True`.

## 10. Time / Space Complexity và so sánh

`L` = độ dài `flowerbed`, `z` = số ô trống.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 0. Brute force | `O(2^z · L)` | `O(L)` | Chỉ để kiểm chứng |
| 1. Tham lam (copy) | `O(L)` | `O(L)` bản sao | Rõ ràng |
| 1'. Tham lam sửa tại chỗ + dừng sớm | `O(L)` | `O(1)` | Bản nộp chính |
| 2. Đệm | `O(L)` | `O(L)` (list mới) | Không phải nghĩ về biên |
| 3. Đếm đoạn 0 | `O(L)` | `O(1)` | Không sửa input, có công thức |

## 11. Bộ test: mẫu đề, biên, đối chiếu brute force

In [13]:
import random

methods = {
    "greedy": can_place_greedy,
    "padding": can_place_padding,
    "count_zeros": can_place_count_zeros,
    "Solution": lambda bed, n: Solution().canPlaceFlowers(bed[:], n),
    "SolutionCountZeros": lambda bed, n: SolutionCountZeros().canPlaceFlowers(bed, n),
}
cases = [
    ("đề 1", [1, 0, 0, 0, 1], 1, True),
    ("đề 2", [1, 0, 0, 0, 1], 2, False),
    ("n = 0 luôn True", [1], 0, True),
    ("1 ô trống", [0], 1, True),
    ("1 ô có hoa", [1], 1, False),
    ("biên trái", [0, 0, 1], 1, True),
    ("biên phải", [1, 0, 0], 1, True),
    ("toàn 0 lẻ", [0, 0, 0, 0, 0], 3, True),
    ("toàn 0 chẵn", [0, 0, 0, 0], 3, False),
    ("kẹp 2 số 0", [1, 0, 0, 1], 1, False),
    ("ví dụ trace", [0, 0, 1, 0, 0, 0, 0, 1, 0, 0], 3, True),
]
for name, bed, n, expected in cases:
    for mname, f in methods.items():
        got = f(bed, n)
        assert got == expected, (name, mname, got)
    print(f"{name:<18} PASS -> {expected}")

đề 1               PASS -> True
đề 2               PASS -> False
n = 0 luôn True    PASS -> True
1 ô trống          PASS -> True
1 ô có hoa         PASS -> False
biên trái          PASS -> True
biên phải          PASS -> True
toàn 0 lẻ          PASS -> True
toàn 0 chẵn        PASS -> False
kẹp 2 số 0         PASS -> False
ví dụ trace        PASS -> True


In [14]:
def random_bed(length, rng):
    """Sinh luồng hoa hợp lệ (không có hai số 1 kề nhau)."""
    bed = []
    for _ in range(length):
        if bed and bed[-1] == 1:
            bed.append(0)
        else:
            bed.append(rng.choice([0, 0, 1]))
    return bed

rng = random.Random(605)
for _ in range(400):
    bed = random_bed(rng.randint(1, 12), rng)
    best = max_new_flowers_bruteforce(bed)
    for n in range(0, len(bed) + 1):
        expected = n <= best
        for mname, f in methods.items():
            assert f(bed, n) == expected, (bed, n, mname)
print("400 luồng hoa ngẫu nhiên × mọi n: khớp brute force — PASS")

400 luồng hoa ngẫu nhiên × mọi n: khớp brute force — PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Không ghi lại việc trồng
Chỉ `count += 1` mà không đặt `bed[i] = 1` → ô kế tiếp tưởng hàng xóm vẫn trống, đếm thừa.

In [15]:
def wrong_no_mark(flowerbed, n):
    count = 0
    for i in range(len(flowerbed)):
        if (flowerbed[i] == 0 and (i == 0 or flowerbed[i - 1] == 0)
                and (i == len(flowerbed) - 1 or flowerbed[i + 1] == 0)):
            count += 1                         # quên flowerbed[i] = 1
    return count >= n

print("[0,0,0], n=3 -> sai:", wrong_no_mark([0, 0, 0], 3), "| đúng:", can_place_greedy([0, 0, 0], 3))

[0,0,0], n=3 -> sai: True | đúng: False


### Lỗi 2: Dùng `bed[i-1]` khi `i = 0`
Python không báo lỗi mà đọc **phần tử cuối**. Ví dụ `[0, 0, 1]`: ô 0 xét "trái" là `bed[-1] = 1` → tưởng không trồng được.

### Lỗi 3: Đọc `bed[i+1]` khi `i` là ô cuối → `IndexError`.

In [16]:
def wrong_negative_index(flowerbed, n):
    bed, count = flowerbed[:], 0
    for i in range(len(bed) - 1):                  # cố tránh IndexError bằng cách bỏ ô cuối...
        if bed[i] == 0 and bed[i - 1] == 0 and bed[i + 1] == 0:   # ...nhưng i=0 đọc bed[-1]
            bed[i] = 1
            count += 1
    return count >= n

print("[0,0,1], n=1 -> sai:", wrong_negative_index([0, 0, 1], 1), "| đúng:", can_place_greedy([0, 0, 1], 1))
print("[1,0,0], n=1 -> sai:", wrong_negative_index([1, 0, 0], 1), "| đúng:", can_place_greedy([1, 0, 0], 1))

[0,0,1], n=1 -> sai: False | đúng: True
[1,0,0], n=1 -> sai: False | đúng: True


### Lỗi 4: Trả `count == n` thay vì `count >= n`
Trồng được **nhiều hơn** `n` vẫn nghĩa là trồng được `n`.

### Lỗi 5: Công thức đoạn 0 dùng chung `(z − 1) // 2` cho cả đoạn biên
`[0, 0, 1]` sẽ ra `0` thay vì `1`. Nhớ "số 0 ảo" ở biên.

## 13. Mẹo nhận dạng pattern và biến thể

**Dấu hiệu tham lam:** quyết định cục bộ trái → phải, "trồng/đặt/chọn được thì làm ngay", và có lập luận "dời về trái không thiệt". Các bài ràng buộc "không kề nhau" trên **một dãy** thường như vậy.

**Kỹ thuật biên:** đệm phần tử ảo (`[0] + a + [0]`), hoặc `i == 0 or ...` dựa vào đoản mạch.

**Biến thể:**
- Khoảng cách tối thiểu `k` giữa hai bông (không chỉ "không kề nhau").
- Luồng hoa **vòng tròn** (ô đầu và ô cuối kề nhau).
- 198. House Robber: cũng "không chọn hai ô kề nhau" nhưng **mỗi ô có giá trị** khác nhau → tham lam sai, cần **quy hoạch động (DP)**.

In [17]:
# Vì sao tham lam KHÔNG dùng được cho House Robber (mỗi nhà có tiền khác nhau):
houses = [2, 9, 3]
greedy_take = houses[0] + houses[2]          # tham lam lấy nhà đầu tiên trước
best = max(houses[0] + houses[2], houses[1])
print("Tham lam lấy từ trái:", greedy_take, "| tối ưu:", best)

Tham lam lấy từ trái: 5 | tối ưu: 9


## 14. Bài tập (TODO – không gọi khi chưa làm)

**Bài 1 — Tính tay.** `flowerbed = [0,1,0,0,0,0,0,1,0,0,0]`. Tối đa trồng thêm bao nhiêu bông?

**Bài 2 — Đoán output.** `(i == 0) or [5, 6][i - 1] == 0` với `i = 0` có lỗi không, trả gì?

**Bài 3 — Viết `max_new_flowers(bed)`** trả về **số bông tối đa** (không cần `n`).

**Bài 4 — Liệt kê vị trí** tham lam sẽ trồng: `planting_positions(bed)` → list chỉ số.

**Bài 5 — Biến thể khoảng cách `k`:** giữa hai bông bất kỳ phải có ít nhất `k` ô trống (`k = 1` là bài gốc). Bài cho `bed` hợp lệ theo `k`. Trả về số bông tối đa trồng thêm.

**Bài 6 — Mini challenge (vòng tròn):** ô đầu và ô cuối coi như kề nhau. Trả về số bông tối đa trồng thêm. (Gợi ý: tách thành hai trường hợp.)

In [18]:
answer_ex1 = None   # TODO
answer_ex2 = None   # TODO


def max_new_flowers(bed):
    # TODO: tham lam hoặc đếm đoạn 0
    pass


def planting_positions(bed):
    # TODO: tham lam, ghi lại các i được trồng
    pass

# Kiểm tra sau khi làm:
# assert max_new_flowers([1, 0, 0, 0, 1]) == 1
# assert planting_positions([0, 0, 1, 0, 0, 0, 0, 1, 0, 0]) == [0, 4, 9]

In [19]:
def max_new_flowers_gap(bed, k):
    # TODO: ô i trồng được nếu k ô bên trái và k ô bên phải (trong phạm vi) đều trống
    pass


def max_new_flowers_circular(bed):
    # TODO: trường hợp mảng 1 ô xử lý riêng; còn lại thử bỏ ô đầu hoặc bỏ ô cuối
    pass

# Kiểm tra sau khi làm:
# assert max_new_flowers_gap([0, 0, 0, 0, 0, 0, 0], 2) == 3
# assert max_new_flowers_circular([0, 0, 0]) == 1

## 15. Gợi ý 3 mức

**Bài 1:** (1) Tách đoạn 0. (2) Đoạn đầu chạm biên `z=1`, đoạn giữa `z=5`, đoạn cuối chạm biên `z=3`. (3) `0 + 2 + 1`.

**Bài 2:** (1) Đoản mạch của `or`. (2) Vế trái đã `True`. (3) Không đọc `[5,6][-1]`.

**Bài 3:** (1) Dùng lại cách tham lam, bỏ `n`. (2) Trả `count`. (3) Hoặc dùng công thức đoạn 0.

**Bài 4:** (1) Tham lam như cũ. (2) Mỗi lần trồng `positions.append(i)`. (3) Làm trên bản sao.

**Bài 5:** (1) Tham lam vẫn đúng (lập luận dời về trái). (2) Kiểm tra `bed[max(0, i-k) : i+k+1]` toàn 0 (dùng `any`). (3) Trồng thì đặt `bed[i] = 1`.

**Bài 6:** (1) Vòng tròn: ô 0 và ô cuối kề nhau nên không thể cùng được trồng; mảng 1 ô xử lý riêng. (2) Chia hai trường hợp: A – **cấm trồng** ô cuối, B – **cấm trồng** ô đầu; mỗi trường hợp chạy tham lam trên dãy thẳng (ô bị cấm vẫn trống, không chặn hàng xóm). Nếu ô đầu đã có hoa thì ô cuối cũng bị cấm (và ngược lại). (3) Đáp án = `max(A, B)`; truyền tập chỉ số bị cấm `locked` vào hàm tham lam và `continue` khi `i in locked`.

## 16. Đáp án tham khảo (chạy kiểm tra)

In [20]:
def ans_max_new_flowers(bed):
    bed = bed[:]
    count = 0
    for i in range(len(bed)):
        if bed[i] == 0 and (i == 0 or bed[i - 1] == 0) and (i == len(bed) - 1 or bed[i + 1] == 0):
            bed[i] = 1
            count += 1
    return count

ans_ex1 = ans_max_new_flowers([0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0])   # 0 + 2 + 1 = 3
i = 0
ans_ex2 = (i == 0) or [5, 6][i - 1] == 0                             # True, không lỗi


def ans_planting_positions(bed):
    bed = bed[:]
    pos = []
    for i in range(len(bed)):
        if bed[i] == 0 and (i == 0 or bed[i - 1] == 0) and (i == len(bed) - 1 or bed[i + 1] == 0):
            bed[i] = 1
            pos.append(i)
    return pos


def ans_max_new_flowers_gap(bed, k):
    bed = bed[:]
    count = 0
    for i in range(len(bed)):
        if not any(bed[max(0, i - k): i + k + 1]):
            bed[i] = 1
            count += 1
    return count


def ans_max_new_flowers_circular(bed):
    L = len(bed)
    if L == 1:
        return 1 - bed[0]

    def linear_with_locked(arr, locked):
        """Tham lam trên dãy thẳng nhưng không được trồng ở các chỉ số trong locked."""
        arr = arr[:]
        cnt = 0
        for i in range(len(arr)):
            if i in locked:
                continue
            if arr[i] == 0 and (i == 0 or arr[i - 1] == 0) and (i == len(arr) - 1 or arr[i + 1] == 0):
                arr[i] = 1
                cnt += 1
        return cnt

    # Vòng tròn: ô 0 và ô L-1 kề nhau. Nếu một trong hai đã có hoa, ô kia không được trồng.
    locked_first = {0} if bed[L - 1] == 1 else set()
    locked_last = {L - 1} if bed[0] == 1 else set()
    # Trường hợp A: cấm trồng ô cuối; trường hợp B: cấm trồng ô đầu. Lấy max.
    a = linear_with_locked(bed, locked_first | {L - 1})
    b_ = linear_with_locked(bed, locked_last | {0})
    return max(a, b_)


def circular_bruteforce(bed):
    L, empties, best = len(bed), [i for i, v in enumerate(bed) if v == 0], 0
    for mask in range(1 << len(empties)):
        t = bed[:]
        for k, p in enumerate(empties):
            if mask >> k & 1:
                t[p] = 1
        if L == 1 or all(not (t[i] and t[(i + 1) % L]) for i in range(L)):
            best = max(best, bin(mask).count("1"))
    return best


print("Bài 1:", ans_ex1, "| Bài 2:", ans_ex2)
assert ans_ex1 == 3 and ans_ex2 is True
assert ans_max_new_flowers([1, 0, 0, 0, 1]) == 1
assert ans_planting_positions([0, 0, 1, 0, 0, 0, 0, 1, 0, 0]) == [0, 4, 9]
assert ans_max_new_flowers_gap([0, 0, 0, 0, 0, 0, 0], 2) == 3
assert all(ans_max_new_flowers_gap(b2, 1) == ans_max_new_flowers(b2) for b2 in ([0], [1, 0, 0], [0, 0, 0, 0]))
assert ans_max_new_flowers_circular([0, 0, 0]) == 1
rng = random.Random(6050)
for _ in range(300):
    b2 = random_bed(rng.randint(1, 10), rng)
    if len(b2) > 1 and b2[0] == 1 and b2[-1] == 1:
        continue                          # không hợp lệ trên vòng tròn
    assert ans_max_new_flowers_circular(b2) == circular_bruteforce(b2), b2
print("Đáp án: tất cả test PASS (vòng tròn đối chiếu brute force 300 lần)")

Bài 1: 3 | Bài 2: True
Đáp án: tất cả test PASS (vòng tròn đối chiếu brute force 300 lần)


## 17. Template nhớ nhanh và checklist submit

```python
class Solution:
    def canPlaceFlowers(self, flowerbed: list[int], n: int) -> bool:
        if n == 0:
            return True
        L = len(flowerbed)
        for i in range(L):
            if (flowerbed[i] == 0
                    and (i == 0 or flowerbed[i - 1] == 0)        # biên trái
                    and (i == L - 1 or flowerbed[i + 1] == 0)):  # biên phải
                flowerbed[i] = 1                                 # GHI LẠI việc trồng
                n -= 1
                if n == 0:
                    return True
        return False
```

Câu nhớ: **"trống – trái trống – phải trống → trồng ngay và đánh dấu"**.

### Checklist
- [ ] `n == 0` trả `True`?
- [ ] Kiểm tra biên bằng `i == 0 or ...` / `i == L-1 or ...` (không đọc `bed[-1]`)?
- [ ] Đặt `flowerbed[i] = 1` sau khi trồng?
- [ ] So sánh `>= n` (hoặc giảm `n` về 0)?
- [ ] Thử `[0]`, `[1]`, `[0,0]`, `[0,0,1]`, `[1,0,0]`, toàn 0 độ dài chẵn/lẻ?